# Phase 9 — Transparent Incident Risk Scoring & Prioritization

**Cyber Threat Decision Support Pipeline — Phase 9**

This research notebook evaluates and visualizes the explainable, calibrated incident risk scoring model:
$$\text{Score} = w_1 \cdot C_{\text{ML}} + w_2 \cdot \text{Sev} + w_3 \cdot S_{\text{vol}} + w_4 \cdot B_{\text{rule}}$$

### Core Principles & Scientific Rigor:
1. **Bounded Composite Score:** $\text{Score} \in [0, 100]$ strictly enforced via convex weights ($w_k \ge 0, \sum w_k = 1.0$).
2. **Additive Sub-Score Attribution:** Every score decomposes into four transparent additive contributions $(w_1 C_{\text{ML}}, w_2 \text{Sev}, w_3 S_{\text{vol}}, w_4 B_{\text{rule}})$.
3. **Zero Data Leakage:** Pre-registered 50/50 incident split ($1,652$ calibration vs $1,644$ report) on the 20% unseen test partition. Weights were calibrated exclusively on the calibration half and committed before scoring the report half.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Locate artifacts
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
risk_dir = root / 'artifacts' / 'risk'

with open(risk_dir / 'risk_weights_v1.json', 'r') as f:
    calib_cfg = json.load(f)

df = pd.read_parquet(risk_dir / 'incident_risk_ranking_v1.parquet')
report_df = df[df['partition'] == 'report'].sort_values(by='risk_score', ascending=False).reset_index(drop=True)

print("=== PHASE 9 CALIBRATION COMMITMENT ===")
print(f"Calibrated Weights: w1={calib_cfg['weights'][0]}, w2={calib_cfg['weights'][1]}, w3={calib_cfg['weights'][2]}, w4={calib_cfg['weights'][3]}")
print(f"Rule Bonus Variant: {calib_cfg['rule_variant']}")
print(f"Reference Volume N_ref (99th %ile): {calib_cfg['N_ref']}")
print(f"Calibration ROC-AUC: {calib_cfg['calibration_auc']:.5f}")
print(f"\nReport Partition: {len(report_df)} incidents ({sum(report_df['incident_label'] == 'attack')} attack, {sum(report_df['incident_label'] == 'false')} false)")

## 1. Risk Score Distribution: True Attack vs. False Alarm Separation

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

attack_scores = report_df[report_df['incident_label'] == 'attack']['risk_score']
false_scores = report_df[report_df['incident_label'] == 'false']['risk_score']

ax.hist(false_scores, bins=25, alpha=0.6, color='#e74c3c', label=f'False Alarms (n={len(false_scores)})', density=True)
ax.hist(attack_scores, bins=25, alpha=0.6, color='#2980b9', label=f'True Attacks (n={len(attack_scores)})', density=True)

ax.set_xlabel('Calibrated Risk Score (0 - 100)', fontweight='bold')
ax.set_ylabel('Density', fontweight='bold')
ax.set_title('Risk Score Distribution on Unseen Report Half (ROC-AUC = 0.9661)', fontsize=12, fontweight='bold')
ax.legend(frameon=True)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 2. ROC Discrimination Comparison (Hybrid vs. Baselines)

In [ ]:
eval_df = report_df[report_df['incident_label'].isin(['attack', 'false'])]
y_true = (eval_df['incident_label'] == 'attack').astype(int).to_numpy()

# Compute ROC curves
fpr_hybrid, tpr_hybrid, _ = roc_curve(y_true, eval_df['risk_score'])
fpr_ml, tpr_ml, _ = roc_curve(y_true, eval_df['c_ml_raw'])
fpr_vol, tpr_vol, _ = roc_curve(y_true, eval_df['s_vol_raw'])
fpr_sev, tpr_sev, _ = roc_curve(y_true, eval_df['sev_raw'])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_hybrid, tpr_hybrid, color='#2c3e50', linewidth=2.5, label=f'Hybrid Calibrated (AUC = {roc_auc_score(y_true, eval_df["risk_score"]):.4f})')
ax.plot(fpr_ml, tpr_ml, color='#e67e22', linestyle='--', label=f'ML Confidence Only (AUC = {roc_auc_score(y_true, eval_df["c_ml_raw"]):.4f})')
ax.plot(fpr_vol, tpr_vol, color='#27ae60', linestyle=':', label=f'Volume Only (AUC = {roc_auc_score(y_true, eval_df["s_vol_raw"]):.4f})')
ax.plot(fpr_sev, tpr_sev, color='#8e44ad', linestyle='-.', label=f'Severity Only (AUC = {roc_auc_score(y_true, eval_df["sev_raw"]):.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', alpha=0.5, label='Random Chance')

ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate (Recall)', fontweight='bold')
ax.set_title('ROC Discrimination Curves on Unseen Report Half', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', frameon=True)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Sub-Score Contribution Breakdown (Top Analyst Triage Queue)

In [ ]:
top15 = report_df.head(15)

fig, ax = plt.subplots(figsize=(10, 5))
indices = np.arange(len(top15))

cml_bar = top15['contrib_cml'].to_numpy()
sev_bar = top15['contrib_sev'].to_numpy()
vol_bar = top15['contrib_svol'].to_numpy()
rule_bar = top15['contrib_brule'].to_numpy()

p1 = ax.bar(indices, cml_bar, label='w1 * ML Conf', color='#3498db')
p2 = ax.bar(indices, sev_bar, bottom=cml_bar, label='w2 * Severity', color='#9b59b6')
p3 = ax.bar(indices, vol_bar, bottom=cml_bar + sev_bar, label='w3 * Volume', color='#2ecc71')
p4 = ax.bar(indices, rule_bar, bottom=cml_bar + sev_bar + vol_bar, label='w4 * Rule Bonus', color='#e67e22')

labels = [f"{row['primary_threat']}\n(ID:{row['incident_id']})" for _, row in top15.iterrows()]
ax.set_xticks(indices)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Additive Risk Score Contribution', fontweight='bold')
ax.set_ylim(0, 105)
ax.set_title('Top-15 Incident Triage Queue: Transparent Additive Sub-Score Breakdown', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Top-20 Priority Incident Triage Display

In [ ]:
display_cols = [
    'rank_in_report', 'incident_id', 'source_day', 'source_ip', 'destination_port', 
    'primary_threat', 'incident_label', 'n_flows', 'risk_score', 
    'contrib_cml', 'contrib_sev', 'contrib_svol', 'contrib_brule'
]
top20_table = report_df[display_cols].head(20)
print("=== TOP-20 ANALYST TRIAGE QUEUE (REPORT HALF) ===")
print(top20_table.to_string(index=False))